# 昼夜合成反转因子计算示例

这个 notebook 模拟用户在当前 `core/` 框架下实际计算一个高频因子的流程。

目标因子：

```text
stk.1d.day_night_reversal
```

计算口径：

1. 用 `stk.1min.close_price` 计算 10:00 到日内最后一分钟的未复权对数收益。
2. 用复权且停牌过滤后的日频开盘价、收盘价计算隔夜跳空。
3. 将日内收益 20 日均值和隔夜跳空 20 日求和合成最终因子。
4. 用 `stk.1d.is_untradable` 剔除过去 20 日窗口内存在不可交易记录的股票。

建议先用较短区间熟悉流程。

## 1. 导入依赖

In [9]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "core" / "__init__.py").exists():
            return path
    raise RuntimeError("找不到包含 core/__init__.py 的项目根目录")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core import DataRouter, FeatureDef, FeatureManager, FeatureStore, get_hf, get_lf
from core.schema import get_freq_step_values

PROJECT_ROOT


PosixPath('/data/home/dingtianxin/factor-operators')

## 2. 设置本次计算参数

`SNAPSHOT_ROOT` 是本次实验的本地物化目录。正式实验可以换成个人路径下的指定 snapshot 路径。

`START` / `END` 控制股票池、交易日轴和数据读取范围。研究范围越长取数计算耗时越长，参考：1年约1.5分钟，10年在chunk=1000下15分钟。

overlap 在 chunk 模式下需要填一个较保守的大数值(比如60)，或者填None自动推断

In [10]:
TARGET_KEY = "stk.1d.day_night_reversal"

SNAPSHOT_ROOT = Path("/tmp/factorops_day_night_reversal_nb")
START = "2015-01-01"
END = "2025-12-31"

INIT_SNAPSHOT = True
OVERWRITE_SNAPSHOT = True

CHUNK_SIZE = 250
OVERLAP = 25 ### 需要确认overlap是否足够！！！
RETURN_ARRAY = False

OUTPUT_CSV = Path("/tmp/day_night_reversal_runner_nb.csv")
PREVIEW_ROWS = 10


## 3. 初始化 Store、Router 和 Manager

真实使用时通常先初始化 snapshot，固定日期轴和资产轴。之后所有数据都会对齐到这个 snapshot。第一次建立研究快照需要初始化，后续可以在前面设置 INIT_SNAPSHOT=False 关闭初始化。

初始化 1年 范围约9秒；10年约 1分钟。

In [11]:
store = FeatureStore(SNAPSHOT_ROOT)

if INIT_SNAPSHOT:
    store.init_snapshot(
        start=START,
        end=END,
        assets=("stk",),
        overwrite=OVERWRITE_SNAPSHOT,
    )

router = DataRouter()
manager = FeatureManager(store, data_router=router)

print("snapshot_root:", SNAPSHOT_ROOT)
print("date axis:", store.get_dates()[0], "->", store.get_dates()[-1], "count=", len(store.get_dates()))
print("stk asset count:", len(store.get_asset_codes("stk")))


snapshot_root: /tmp/factorops_day_night_reversal_nb
date axis: 20150105 -> 20251231 count= 2674
stk asset count: 5415


## 4. 看一眼本因子需要的数据源

第一次写因子时建议看一下，确认字段是否能被 `DataRouter` 解析到预期的数据源(即字段名有没有写对)。

比如：

- `stk.1min.close_price` 应该走 `MinuteParquet`。
- `stk.1d.is_untradable` 应该走 `SmartQuant.Untradable`。

In [12]:
required_keys = [
    "stk.1min.close_price",
    "stk.1d.OpenPrice",
    "stk.1d.ClosePrice",
    "stk.1d.IfSuspended",
    "stk.1d.adj_factor",
    "stk.1d.is_untradable",
]

source_rows = []
for key in required_keys:
    spec = router.resolve_source(key)
    source_rows.append(
        {
            "key": key,
            "source": spec.source,
            "table": spec.table,
            "field": spec.field,
            "params": spec.params,
        }
    )

source_df = pd.DataFrame(source_rows)
source_df


,key,source,table,field,params
0,stk.1min.close_price,MinuteParquet,/data/cephfs/minute/one_minute_stat/{date}.par...,close_price,"{'data_type': 'one_minute_stat', 'path_templat..."
1,stk.1d.OpenPrice,ReturnDaily,SmartQuant.ReturnDaily,OpenPrice,{}
2,stk.1d.ClosePrice,ReturnDaily,SmartQuant.ReturnDaily,ClosePrice,{}
3,stk.1d.IfSuspended,ReturnDaily,SmartQuant.ReturnDaily,IfSuspended,{}
4,stk.1d.adj_factor,AdjustFactor,JYDB.DZ_AdjustingFactor,adj_factor,{}
5,stk.1d.is_untradable,Untradable,SmartQuant.Untradable,is_untradable,{}


## 5. 找到 10:00 在 1min step 轴上的位置

框架公式里的 `get_step(..., step=...)` 使用的是 step 位置，不是 `start_time` 标签。

当前 1min 真实 step 轴为：

```text
930..1129 + 1300..1456
```

因此需要先把 `1000` 转成位置。

In [13]:
step_values = get_freq_step_values("1min")
pos_1000 = int(np.where(step_values == 1000)[0][0])

print("1min step count:", len(step_values))
print("first step:", int(step_values[0]))
print("last step:", int(step_values[-1]))
print("1000 position:", pos_1000)
print("head:", step_values[:8].tolist())
print("tail:", step_values[-8:].tolist())


1min step count: 237
first step: 930
last step: 1456
1000 position: 30
head: [930, 931, 932, 933, 934, 935, 936, 937]
tail: [1449, 1450, 1451, 1452, 1453, 1454, 1455, 1456]


## 6. 定义因子

下面只使用用户侧公开 API：

- `get_hf` 生成分钟字段定义。
- `get_lf` 生成日频字段定义，并处理复权和停牌过滤。
- `FeatureDef.from_key` 定义中间公式和最终因子。

这些定义先注册到 `FeatureManager`，真正计算发生在后面的 `manager.materialize(...)`。

In [14]:
tradability_mask = """
    equal(
        ts_sum(
            where(equal(stk.1d.is_untradable, 1), 1, 0),
            window=20,
            min_periods=1
        ),
        0
    )
    """


feature_defs = [
    get_hf(
        "close_price",
        asset="stk",
        freq="1min",
        alias="min_close",
        materialize=False,
        overwrite=True,
    ),
    FeatureDef.from_key(
        "stk.1d.intra_1000_close_logret",
        formula=f"""
        ln(
            step_last(
                divide(
                    stk.1min.close_price,
                    get_step(stk.1min.close_price, step={pos_1000})
                )
            )
        )
        """,
        materialize=False,
        overwrite=True,
        metadata={"factor_part": "intraday_logret"},
    ),
    FeatureDef.from_key(
        "stk.1d.intraday_return_20",
        formula="""
        ts_mean(
            stk.1d.intra_1000_close_logret,
            window=20,
            min_periods=1
        )
        """,
        materialize=False,
        overwrite=True,
    ),
    get_lf(
        "OpenPrice",
        asset="stk",
        freq="1d",
        name="OpenPrice_Adj_Sus",
        if_adj=True,
        if_sus=True,
        materialize=False,
        overwrite=True,
    ),
    get_lf(
        "ClosePrice",
        asset="stk",
        freq="1d",
        name="ClosePrice_Adj_Sus",
        if_adj=True,
        if_sus=True,
        materialize=False,
        overwrite=True,
    ),
    get_lf(
        "OpenPrice",
        asset="stk",
        freq="1d",
        name="OpenPrice_Sus",
        if_adj=False,
        if_sus=True,
        materialize=False,
        overwrite=True,
    ),
    get_lf(
        "ClosePrice",
        asset="stk",
        freq="1d",
        name="ClosePrice_Sus",
        if_adj=False,
        if_sus=True,
        materialize=False,
        overwrite=True,
    ),
    FeatureDef.from_key(
        "stk.1d.overnight_gap",
        formula="""
        abs(
            ln(
                divide(
                    stk.1d.OpenPrice_Sus,
                    delay(ts_ffill(stk.1d.ClosePrice_Sus), periods=1, axis=0)
                )
            )
        )
        """,
        materialize=False,
        overwrite=True,
        metadata={"factor_part": "overnight_gap"},
    ),
    FeatureDef.from_key(
        "stk.1d.overnight_gap_20",
        formula="""
        ts_sum(
            ts_ffill(stk.1d.overnight_gap),
            window=20,
            min_periods=10
        )
        """,
        materialize=False,
        overwrite=True,
    ),
    FeatureDef.from_key(
        TARGET_KEY,
        formula="""
        -(
            0.6 * stk.1d.intraday_return_20
            + 0.4 * stk.1d.overnight_gap_20
        )
        """,
        output_mask=None,
        materialize=True,
        overwrite=True,
        metadata={"factor": "day_night_reversal"},
    ),
]


## 7. 注册定义并查看依赖

注册后，`FeatureManager` 会解析公式中的依赖。下面的表可以帮助确认最终因子依赖是否符合预期。

以及目前 在分块模式下 必须只对最终因子 填`materialize=True`

In [15]:
registered_defs = [manager.register(feature_def, overwrite=True) for feature_def in feature_defs]

definition_df = pd.DataFrame(
    [
        {
            "key": feature_def.key,
            "materialize": feature_def.materialize,
            "dependencies": ", ".join(feature_def.dependencies),
        }
        for feature_def in registered_defs
    ]
)

definition_df


,key,materialize,dependencies
0,stk.1min.close_price,False,stk.1min.close_price
1,stk.1d.intra_1000_close_logret,False,stk.1min.close_price
2,stk.1d.intraday_return_20,False,stk.1d.intra_1000_close_logret
3,stk.1d.OpenPrice_Adj_Sus,False,"stk.1d.OpenPrice, stk.1d.adj_factor, stk.1d.If..."
4,stk.1d.ClosePrice_Adj_Sus,False,"stk.1d.ClosePrice, stk.1d.adj_factor, stk.1d.I..."
5,stk.1d.OpenPrice_Sus,False,"stk.1d.OpenPrice, stk.1d.IfSuspended"
6,stk.1d.ClosePrice_Sus,False,"stk.1d.ClosePrice, stk.1d.IfSuspended"
7,stk.1d.overnight_gap,False,"stk.1d.OpenPrice_Sus, stk.1d.ClosePrice_Sus"
8,stk.1d.overnight_gap_20,False,stk.1d.overnight_gap
9,stk.1d.day_night_reversal,True,"stk.1d.intraday_return_20, stk.1d.overnight_ga..."


## 8. 物化最终因子

这里使用 chunk 模式，避免一次性读取过多分钟数据。

`OVERLAP=20` 用来覆盖 `delay(1)` 和 20 日 rolling 的 warmup。

In [16]:
result = manager.materialize(
    TARGET_KEY,
    overwrite=True,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
    return_array=RETURN_ARRAY,
)

print("materialize returned:", type(result).__name__ if result is not None else None)


materialize returned: None


## 结果数组可以通过 store.load_feature()读取

1年 0.3s

In [17]:
factor = store.load_feature(TARGET_KEY)

## 9. 查看覆盖率



In [18]:
factor = store.load_feature(TARGET_KEY)
values = factor.values[:, :, 0]
finite = np.isfinite(values)

coverage_df = pd.DataFrame(
    {
        "DataDate": factor.space.dates.astype(str),
        "coverage": finite.sum(axis=1) / values.shape[1],
        "finite_count": finite.sum(axis=1),
    }
)

coverage_df.tail(PREVIEW_ROWS)


,DataDate,coverage,finite_count
2664,20251218,0.954571,5169
2665,20251219,0.954755,5170
2666,20251222,0.954755,5170
2667,20251223,0.954571,5169
2668,20251224,0.954755,5170
2669,20251225,0.954755,5170
2670,20251226,0.954755,5170
2671,20251229,0.954755,5170
2672,20251230,0.955125,5172
2673,20251231,0.955309,5173


## 10. 查看最新一期截面分布

In [19]:
latest = pd.Series(values[-1], name=str(factor.space.dates[-1]))
latest.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])


count    5173.000000
mean       -0.047161
std         0.041607
min        -0.810670
1%         -0.207676
5%         -0.121856
50%        -0.034911
95%        -0.013273
99%        -0.008350
max        -0.003058
Name: 20251231, dtype: float64

## 11. 导出最后一期 因子值 dataframe 格式结果

框架内部资产轴是 `InnerCode`。如果要交给下游同事或信号表，通常需要转回 `SecuCode`。

In [20]:
target_date = str(factor.space.dates[-1])
code_map = store.get_code_map("stk")
today_map = code_map[code_map["DataDate"].astype(str) == target_date]
inner_to_secu = dict(zip(today_map["InnerCode"], today_map["SecuCode"]))

runner = pd.DataFrame(
    {
        "runner_date": target_date,
        "InnerCode": factor.space.codes,
        "SecuCode": [inner_to_secu.get(code) for code in factor.space.codes],
        "runner_value": values[-1],
    }
)
runner = runner.dropna(subset=["SecuCode", "runner_value"])
runner = runner[["runner_date", "SecuCode", "runner_value"]]

if OUTPUT_CSV is not None:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    runner.to_csv(OUTPUT_CSV, index=False)
    print("saved:", OUTPUT_CSV)

runner.head(PREVIEW_ROWS)


saved: /tmp/day_night_reversal_runner_nb.csv


,runner_date,SecuCode,runner_value
0,20251231,000001,-0.006713
1,20251231,000002,-0.065600
2,20251231,000004,-0.047982
4,20251231,000006,-0.049303
5,20251231,000007,-0.066466
6,20251231,000008,-0.019773
7,20251231,000009,-0.027037
8,20251231,000010,-0.033772
9,20251231,000011,-0.017513
10,20251231,000012,-0.019884
